<a href="https://colab.research.google.com/github/Nawat1124/wind-turbine-scada-anomaly-detection/blob/main/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# =========================================================
# Paths
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3")

RECON_DIR = PROJECT_DIR
DATA_DIR = PROJECT_DIR / "data"
MODEL_READY_DIR = PROJECT_DIR / "model_ready_simple"

MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)

DF_RECON_PATH = RECON_DIR / "df_reconstructed.parquet"
LINEAGE_PATH = RECON_DIR / "row_lineage_map.parquet"
EVENT_MAPPING_PATH = RECON_DIR / "event_mapping.parquet"

FEATURE_DESCRIPTION_PATH = DATA_DIR / "feature_description.csv"
EVENT_INFO_PATH = DATA_DIR / "event_info_v6.csv"

In [ ]:
# =========================================================
# 1. Load v3 reconstructed artifacts
# =========================================================

df_reconstructed = pd.read_parquet(DF_RECON_PATH)
row_lineage_map = pd.read_parquet(LINEAGE_PATH)
event_mapping = pd.read_parquet(EVENT_MAPPING_PATH)

feature_description = pd.read_csv(FEATURE_DESCRIPTION_PATH)
event_info = pd.read_csv(EVENT_INFO_PATH)

# Standardize time columns
if "reconstructed_time" in df_reconstructed.columns:
    df_reconstructed["reconstructed_time"] = pd.to_datetime(df_reconstructed["reconstructed_time"])

if "reconstructed_time" in row_lineage_map.columns:
    row_lineage_map["reconstructed_time"] = pd.to_datetime(row_lineage_map["reconstructed_time"])

if "start_time" in event_mapping.columns:
    event_mapping["start_time"] = pd.to_datetime(event_mapping["start_time"])

if "end_time" in event_mapping.columns:
    event_mapping["end_time"] = pd.to_datetime(event_mapping["end_time"])

print("df_reconstructed:", df_reconstructed.shape)
print("row_lineage_map:", row_lineage_map.shape)
print("event_mapping:", event_mapping.shape)

df_reconstructed: (560070, 261)
row_lineage_map: (859065, 6)
event_mapping: (15, 15)


In [ ]:
# =========================================================
# 2. Basic sanity checks
# =========================================================

required_df_cols = [
    "asset_id",
    "physical_order_index",
    "reconstructed_time",
    "train_test",
    "status_type_id",
]

required_lineage_cols = [
    "asset_id",
    "source_session_id",
    "source_pos",
    "physical_order_index",
    "reconstructed_time",
    "train_test",
]

missing_df_cols = [c for c in required_df_cols if c not in df_reconstructed.columns]
missing_lineage_cols = [c for c in required_lineage_cols if c not in row_lineage_map.columns]

assert not missing_df_cols, f"Missing columns in df_reconstructed: {missing_df_cols}"
assert not missing_lineage_cols, f"Missing columns in row_lineage_map: {missing_lineage_cols}"

# row_lineage_map v3 should not require status_type_id
assert "status_type_id" not in row_lineage_map.columns, "row_lineage_map should not contain status_type_id in v3."

# One reconstructed physical row should be unique within each asset
dup_physical = df_reconstructed.duplicated(["asset_id", "physical_order_index"]).sum()
assert dup_physical == 0, f"Duplicated reconstructed physical rows: {dup_physical}"

# Lineage rows should map to existing reconstructed physical rows
physical_keys = df_reconstructed[["asset_id", "physical_order_index"]].drop_duplicates()
lineage_keys = row_lineage_map[["asset_id", "physical_order_index"]].drop_duplicates()

missing_lineage_keys = (
    lineage_keys
    .merge(physical_keys, on=["asset_id", "physical_order_index"], how="left", indicator=True)
    .query("_merge == 'left_only'")
)

assert len(missing_lineage_keys) == 0, f"Lineage contains keys not found in df_reconstructed: {len(missing_lineage_keys)}"

print("Basic checks passed.")
print("Unique reconstructed physical rows:", len(physical_keys))
print("Unique lineage physical rows:", len(lineage_keys))

Basic checks passed.
Unique reconstructed physical rows: 560070
Unique lineage physical rows: 560070


In [ ]:
# =========================================================
# 3. Remove confirmed dataset-specific glitch rows
# =========================================================
# This removes the manually confirmed CARE Wind Farm B data glitch.
# Do not reindex physical_order_index after dropping these rows.

d = df_reconstructed.copy()
lineage = row_lineage_map.copy()

glitch_mask = (
    (d["asset_id"] == 2)
    & (d["reconstructed_time"] >= pd.Timestamp("2025-07-02 10:40:00"))
    & (d["reconstructed_time"] <= pd.Timestamp("2025-07-02 12:10:00"))
)

glitch_keys = d.loc[glitch_mask, ["asset_id", "physical_order_index"]].drop_duplicates()

print("Confirmed glitch rows in df_reconstructed:", int(glitch_mask.sum()))
print("Confirmed glitch physical keys:", len(glitch_keys))

if len(glitch_keys) > 0:
    d = (
        d.merge(glitch_keys.assign(_drop_glitch=True),
                on=["asset_id", "physical_order_index"],
                how="left")
         .query("_drop_glitch != True")
         .drop(columns=["_drop_glitch"])
         .copy()
    )

    lineage = (
        lineage.merge(glitch_keys.assign(_drop_glitch=True),
                      on=["asset_id", "physical_order_index"],
                      how="left")
               .query("_drop_glitch != True")
               .drop(columns=["_drop_glitch"])
               .copy()
    )

print("After glitch removal:")
print("d:", d.shape)
print("lineage:", lineage.shape)

Confirmed glitch rows in df_reconstructed: 10
Confirmed glitch physical keys: 10
After glitch removal:
d: (560060, 261)
lineage: (859055, 6)


In [ ]:
# =========================================================
# 4. Angle encoding
# =========================================================
# sensor_4_avg and sensor_21_avg are treated as circular angles.
# sensor_10_avg is kept as raw because it is not a full 0-360 circular angle.

ANGLE_COLS = ["sensor_4_avg", "sensor_21_avg"]

for col in ANGLE_COLS:
    if col in d.columns:
        radians = np.deg2rad(d[col].astype(float))
        d[f"{col}_sin"] = np.sin(radians)
        d[f"{col}_cos"] = np.cos(radians)

# Raw circular angle columns are removed from model candidates after encoding.
angle_raw_cols_to_drop = [c for c in ANGLE_COLS if c in d.columns]
d = d.drop(columns=angle_raw_cols_to_drop)

print("Added angle-encoded columns:")
print([c for c in d.columns if c.endswith("_sin") or c.endswith("_cos")])
print("Dropped raw circular angle columns:", angle_raw_cols_to_drop)
print("d:", d.shape)

Added angle-encoded columns:
['sensor_4_avg_sin', 'sensor_4_avg_cos', 'sensor_21_avg_sin', 'sensor_21_avg_cos']
Dropped raw circular angle columns: ['sensor_4_avg', 'sensor_21_avg']
d: (560060, 263)


In [ ]:
# =========================================================
# 5. Drop all-zero / constant / unusable variables
# =========================================================

META_COLS = {
    "asset_id",
    "source_session_id",
    "source_pos",
    "physical_order_index",
    "reconstructed_time",
    "time_stamp",
    "train_test",
    "status_type_id",
}

numeric_cols = d.select_dtypes(include=[np.number]).columns.tolist()
candidate_feature_cols = [c for c in numeric_cols if c not in META_COLS]

# Drop all-zero columns
all_zero_cols = [
    c for c in candidate_feature_cols
    if d[c].notna().any() and np.isclose(d[c].fillna(0).abs().sum(), 0)
]

# Drop constant columns
constant_cols = [
    c for c in candidate_feature_cols
    if d[c].nunique(dropna=True) <= 1
]

# Drop angle std columns if present
angle_std_cols = [
    c for c in d.columns
    if c in ["sensor_4_std", "sensor_21_std"]
]

drop_unusable_cols = sorted(set(all_zero_cols + constant_cols + angle_std_cols))
d = d.drop(columns=drop_unusable_cols)

print("Dropped all-zero columns:", all_zero_cols)
print("Dropped constant columns:", constant_cols)
print("Dropped angle std columns:", angle_std_cols)
print("Total dropped unusable columns:", len(drop_unusable_cols))
print("d:", d.shape)

Dropped all-zero columns: ['sensor_0_max', 'sensor_0_min', 'sensor_0_std', 'sensor_1_max', 'sensor_1_min', 'sensor_1_std', 'sensor_2_max', 'sensor_2_min', 'sensor_2_std', 'sensor_3_max', 'sensor_3_min', 'sensor_3_std', 'sensor_5_max', 'sensor_5_min', 'sensor_5_std', 'sensor_9_max', 'sensor_9_min', 'sensor_9_std']
Dropped constant columns: ['sensor_0_max', 'sensor_0_min', 'sensor_0_std', 'sensor_1_max', 'sensor_1_min', 'sensor_1_std', 'sensor_2_max', 'sensor_2_min', 'sensor_2_std', 'sensor_3_max', 'sensor_3_min', 'sensor_3_std', 'sensor_5_max', 'sensor_5_min', 'sensor_5_std', 'sensor_9_max', 'sensor_9_min', 'sensor_9_std']
Dropped angle std columns: ['sensor_4_std', 'sensor_21_std']
Total dropped unusable columns: 20
d: (560060, 243)


In [ ]:
# =========================================================
# 6. Remove counter-like sensor_0–3 family
# =========================================================
# These variables were excluded because they behave like unstable counters
# rather than stable physical-state variables.

counter_sensor_prefixes = [f"sensor_{i}_" for i in range(4)]

counter_cols = [
    c for c in d.columns
    if any(c.startswith(prefix) for prefix in counter_sensor_prefixes)
]

d = d.drop(columns=counter_cols)

print("Dropped counter-like columns:", counter_cols)
print("Number of dropped counter-like columns:", len(counter_cols))
print("d:", d.shape)

Dropped counter-like columns: ['sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'sensor_3_avg']
Number of dropped counter-like columns: 4
d: (560060, 239)


In [ ]:
# =========================================================
# 7. Add never_prediction_physical_row
# =========================================================
# A reconstructed physical row is safe from prediction leakage only if
# none of its source rows came from a prediction segment.

lineage["is_prediction_source"] = (
    lineage["train_test"]
    .astype(str)
    .str.lower()
    .str.contains("prediction")
)

prediction_flag = (
    lineage
    .groupby(["asset_id", "physical_order_index"], as_index=False)["is_prediction_source"]
    .max()
    .rename(columns={"is_prediction_source": "ever_prediction_physical_row"})
)

d = d.merge(
    prediction_flag,
    on=["asset_id", "physical_order_index"],
    how="left"
)

d["ever_prediction_physical_row"] = d["ever_prediction_physical_row"].fillna(False).astype(bool)
d["never_prediction_physical_row"] = ~d["ever_prediction_physical_row"]

normal_mask = d["status_type_id"].isin([0, 2])
safe_normal_mask = normal_mask & d["never_prediction_physical_row"]

print("never_prediction_physical_row counts:")
print(d["never_prediction_physical_row"].value_counts(dropna=False))

print("\nNormal rows:", int(normal_mask.sum()))
print("Safe normal rows:", int(safe_normal_mask.sum()))
print("d:", d.shape)

never_prediction_physical_row counts:
never_prediction_physical_row
True     490957
False     69103
Name: count, dtype: int64

Normal rows: 507393
Safe normal rows: 443248
d: (560060, 241)


In [ ]:
# =========================================================
# 8. Define final model feature list
# =========================================================
# LSTM input features:
# - avg variables
# - angle sin/cos variables
# - remove highly redundant avg variables
# - remove counter-like variables already dropped earlier

AVG_FAMILY = {
    "power_energy_output": [
        "reactive_power_11_avg", "sensor_26_avg", "power_58_avg",
        "power_62_avg", "sensor_7_avg"
    ],
    "electrical_grid_current_voltage": [
        "sensor_13_avg", "sensor_14_avg", "sensor_15_avg",
        "sensor_16_avg", "sensor_17_avg", "sensor_18_avg",
        "sensor_23_avg", "sensor_24_avg", "sensor_27_avg",
        "sensor_28_avg", "sensor_29_avg", "sensor_30_avg"
    ],
    "wind_orientation_pitch_control": [
        "sensor_4_avg_sin", "sensor_4_avg_cos", "sensor_6_avg", "sensor_10_avg",
        "sensor_21_avg_sin", "sensor_21_avg_cos",
        "wind_speed_59_avg", "wind_speed_60_avg", "wind_speed_61_avg"
    ],
    "drivetrain_speed_torque": [
        "sensor_5_avg", "sensor_12_avg", "sensor_19_avg",
        "sensor_20_avg", "sensor_25_avg", "sensor_57_avg"
    ],
    "ambient_generator_motor_thermal": [
        "sensor_8_avg", "sensor_22_avg",
        "sensor_32_avg", "sensor_33_avg",
        "sensor_48_avg", "sensor_49_avg", "sensor_50_avg", "sensor_53_avg"
    ],
    "gearbox_bearing_oil_thermal": [
        "sensor_34_avg", "sensor_35_avg", "sensor_36_avg",
        "sensor_37_avg", "sensor_38_avg", "sensor_39_avg",
        "sensor_51_avg", "sensor_52_avg"
    ],
    "transformer_thermal": [
        "sensor_40_avg", "sensor_41_avg", "sensor_42_avg", "sensor_43_avg",
        "sensor_44_avg", "sensor_45_avg", "sensor_46_avg", "sensor_47_avg",
        "sensor_31_avg"
    ],
    "vibration_structural": [
        "sensor_9_avg", "sensor_54_avg", "sensor_55_avg", "sensor_56_avg"
    ],
}

DROP_REDUNDANT_AVG_COLS = [
    "sensor_16_avg",
    "sensor_17_avg",
    "sensor_18_avg",
    "sensor_13_avg",
    "sensor_14_avg",
    "sensor_15_avg",
    "sensor_19_avg",
    "sensor_20_avg",
    "sensor_26_avg",
    "sensor_57_avg",
    "wind_speed_59_avg",
    "sensor_6_avg",
]

MODEL_FEATURE_COLS = [
    c for c in d.columns
    if (
        c.endswith("_avg")
        or c in [
            "sensor_4_avg_sin", "sensor_4_avg_cos",
            "sensor_21_avg_sin", "sensor_21_avg_cos",
        ]
    )
    and c not in DROP_REDUNDANT_AVG_COLS
]

MODEL_FEATURE_COLS = sorted(MODEL_FEATURE_COLS)

# Keep old name for compatibility with existing model notebooks.
KMEANS_COLS = MODEL_FEATURE_COLS

missing_values = d[MODEL_FEATURE_COLS].isna().sum().sum()
assert missing_values == 0, f"Model features contain missing values: {missing_values}"

print("Final model feature count:", len(MODEL_FEATURE_COLS))
print(MODEL_FEATURE_COLS)

Final model feature count: 49
['power_58_avg', 'power_62_avg', 'reactive_power_11_avg', 'sensor_10_avg', 'sensor_12_avg', 'sensor_21_avg_cos', 'sensor_21_avg_sin', 'sensor_22_avg', 'sensor_23_avg', 'sensor_24_avg', 'sensor_25_avg', 'sensor_27_avg', 'sensor_28_avg', 'sensor_29_avg', 'sensor_30_avg', 'sensor_31_avg', 'sensor_32_avg', 'sensor_33_avg', 'sensor_34_avg', 'sensor_35_avg', 'sensor_36_avg', 'sensor_37_avg', 'sensor_38_avg', 'sensor_39_avg', 'sensor_40_avg', 'sensor_41_avg', 'sensor_42_avg', 'sensor_43_avg', 'sensor_44_avg', 'sensor_45_avg', 'sensor_46_avg', 'sensor_47_avg', 'sensor_48_avg', 'sensor_49_avg', 'sensor_4_avg_cos', 'sensor_4_avg_sin', 'sensor_50_avg', 'sensor_51_avg', 'sensor_52_avg', 'sensor_53_avg', 'sensor_54_avg', 'sensor_55_avg', 'sensor_56_avg', 'sensor_5_avg', 'sensor_7_avg', 'sensor_8_avg', 'sensor_9_avg', 'wind_speed_60_avg', 'wind_speed_61_avg']


In [ ]:
# =========================================================
# 9. Train / validation / embargo split
# split_code:
# -1 = not eligible
#  0 = train
#  1 = validation
#  2 = embargo
# =========================================================

TIME_COL = "reconstructed_time"
ORDER_COL = "physical_order_index"

d[TIME_COL] = pd.to_datetime(d[TIME_COL])

VAL_RATIO_PER_QUARTER = 0.10
N_FOLDS = 4
EMBARGO_HOURS = 4
EMBARGO_DELTA = pd.Timedelta(hours=EMBARGO_HOURS)

d["split_code"] = np.int8(-1)
d["val_fold"] = np.int8(0)

safe_normal_mask = (
    d["never_prediction_physical_row"].fillna(False)
    & d["status_type_id"].isin([0, 2])
)

d.loc[safe_normal_mask, "split_code"] = np.int8(0)

for aid, g in d.loc[safe_normal_mask].groupby("asset_id"):
    pos = g.sort_values(ORDER_COL).index.to_numpy()
    quarters = np.array_split(pos, N_FOLDS)

    for fold_id, q_pos in enumerate(quarters, start=1):
        n_val = int(round(len(q_pos) * VAL_RATIO_PER_QUARTER))
        if n_val < 1:
            continue

        start = (len(q_pos) - n_val) // 2
        val_pos = q_pos[start:start + n_val]

        d.loc[val_pos, "split_code"] = np.int8(1)
        d.loc[val_pos, "val_fold"] = np.int8(fold_id)

        val_start = d.loc[val_pos, TIME_COL].min()
        val_end = d.loc[val_pos, TIME_COL].max()

        emb_mask = (
            d["asset_id"].eq(aid)
            & safe_normal_mask
            & d["split_code"].eq(0)
            & (
                d[TIME_COL].between(val_start - EMBARGO_DELTA, val_start, inclusive="left")
                | d[TIME_COL].between(val_end, val_end + EMBARGO_DELTA, inclusive="right")
            )
        )

        d.loc[emb_mask, "split_code"] = np.int8(2)

print("split_code counts:")
print(d["split_code"].value_counts().sort_index())

print("\nval_fold counts:")
print(d.loc[d["split_code"].eq(1), "val_fold"].value_counts().sort_index())

split_code counts:
split_code
-1    116812
 0    397198
 1     44327
 2      1723
Name: count, dtype: int64

val_fold counts:
val_fold
1    11083
2    11082
3    11081
4    11081
Name: count, dtype: int64


In [ ]:
# =========================================================
# 10. Fit KMeans and PCA metadata models
# =========================================================
# Both are fitted only on train safe normal rows.
# They are metadata, not LSTM input features.

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

RANDOM_STATE = 42
SAMPLE_N = 200_000
N_CLUSTERS = 6
N_PC = 10

train_pos = np.flatnonzero(d["split_code"].to_numpy() == 0)

if len(train_pos) > SAMPLE_N:
    rng = np.random.default_rng(RANDOM_STATE)
    fit_pos = rng.choice(train_pos, size=SAMPLE_N, replace=False)
    fit_pos = np.sort(fit_pos)
else:
    fit_pos = train_pos

X_fit = d.loc[fit_pos, MODEL_FEATURE_COLS].to_numpy(dtype=np.float32)

print("Rows used to fit metadata models:", X_fit.shape[0])
print("Feature count:", X_fit.shape[1])

# ----- KMeans -----
kmeans_scaler = StandardScaler()
X_fit_km = kmeans_scaler.fit_transform(X_fit)

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=4096,
    random_state=RANDOM_STATE,
    n_init="auto"
)

kmeans.fit(X_fit_km)

# ----- PCA -----
pca_scaler = StandardScaler()
X_fit_pca = pca_scaler.fit_transform(X_fit)

pca = PCA(n_components=N_PC, random_state=RANDOM_STATE)
pca.fit(X_fit_pca)

print("KMeans fitted.")
print("PCA fitted.")
print("PCA cumulative variance:", np.cumsum(pca.explained_variance_ratio_)[-1])

Rows used to fit metadata models: 200000
Feature count: 49
KMeans fitted.
PCA fitted.
PCA cumulative variance: 0.84601504


In [ ]:
# =========================================================
# 11. Add KMeans and PCA metadata to all rows
# =========================================================

X_all = d[MODEL_FEATURE_COLS].to_numpy(dtype=np.float32)

# ----- KMeans metadata -----
X_all_km = kmeans_scaler.transform(X_all)
kmeans_state = kmeans.predict(X_all_km)

d["kmeans_state"] = kmeans_state.astype(np.int8)

centers = kmeans.cluster_centers_
d["kmeans_center_dist"] = np.linalg.norm(
    X_all_km - centers[kmeans_state],
    axis=1
).astype(np.float32)

# ----- PCA metadata -----
X_all_pca = pca_scaler.transform(X_all)
pc_all = pca.transform(X_all_pca)

for i in range(N_PC):
    d[f"pca_PC{i+1}"] = pc_all[:, i].astype(np.float32)

X_recon_pca = pca.inverse_transform(pc_all)

d["pca_residual_after_PC10"] = np.sqrt(
    np.mean((X_all_pca - X_recon_pca) ** 2, axis=1)
).astype(np.float32)

PCA_META_COLS = [f"pca_PC{i+1}" for i in range(N_PC)] + ["pca_residual_after_PC10"]
KMEANS_META_COLS = ["kmeans_state", "kmeans_center_dist"]

print("KMeans state counts:")
print(d["kmeans_state"].value_counts().sort_index())

print("\nPCA metadata columns:")
print(PCA_META_COLS)

print("\nPCA residual summary:")
print(d["pca_residual_after_PC10"].describe())

KMeans state counts:
kmeans_state
0     89308
1     83974
2    128555
3    101994
4     93023
5     63206
Name: count, dtype: int64

PCA metadata columns:
['pca_PC1', 'pca_PC2', 'pca_PC3', 'pca_PC4', 'pca_PC5', 'pca_PC6', 'pca_PC7', 'pca_PC8', 'pca_PC9', 'pca_PC10', 'pca_residual_after_PC10']

PCA residual summary:
count    560060.000000
mean          0.484256
std           2.196881
min           0.111204
25%           0.277987
50%           0.345462
75%           0.444744
max          80.043724
Name: pca_residual_after_PC10, dtype: float64


In [ ]:
# =========================================================
# 12. Build final model-ready dataframe
# =========================================================

META_COLS = [
    "asset_id",
    "physical_order_index",
    "status_type_id",
    "reconstructed_time",
    "never_prediction_physical_row",
    "split_code",
    "val_fold",
]

missing_meta = [c for c in META_COLS if c not in d.columns]
assert not missing_meta, f"Missing metadata columns: {missing_meta}"

df_model = d[
    META_COLS
    + MODEL_FEATURE_COLS
    + PCA_META_COLS
    + KMEANS_META_COLS
].copy()

df_model = df_model.sort_values(
    ["asset_id", "physical_order_index"]
).reset_index(drop=True)

assert df_model[MODEL_FEATURE_COLS].isna().sum().sum() == 0
assert df_model[PCA_META_COLS].isna().sum().sum() == 0
assert df_model[KMEANS_META_COLS].isna().sum().sum() == 0

print("df_model:", df_model.shape)
print("Model input feature count:", len(MODEL_FEATURE_COLS))
print("PCA metadata count:", len(PCA_META_COLS))
print("KMeans metadata count:", len(KMEANS_META_COLS))

df_model: (560060, 69)
Model input feature count: 49
PCA metadata count: 11
KMeans metadata count: 2

Split counts:
split_code
-1    116812
 0    397198
 1     44327
 2      1723
Name: count, dtype: int64


In [ ]:
# =========================================================
# 13. Add prediction event IDs
# =========================================================
# source_session_id is treated as event_id.
# This is metadata only, not model input.

pred = lineage.loc[
    lineage["train_test"].astype(str).str.lower().eq("prediction"),
    ["asset_id", "physical_order_index", "source_session_id"]
].drop_duplicates()

pred = pred.rename(columns={"source_session_id": "event_id"})

pred["event_id"] = (
    pred["event_id"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(np.int16)
)

pred = pred.sort_values(["asset_id", "physical_order_index", "event_id"])

pred["rank"] = (
    pred.groupby(["asset_id", "physical_order_index"])
        .cumcount()
        .add(1)
)

wide = (
    pred.pivot(
        index=["asset_id", "physical_order_index"],
        columns="rank",
        values="event_id"
    )
    .reset_index()
)

wide.columns.name = None

wide = wide.rename(columns={
    1: "pred_event_id_1",
    2: "pred_event_id_2",
})

if "pred_event_id_1" not in wide.columns:
    wide["pred_event_id_1"] = -1

if "pred_event_id_2" not in wide.columns:
    wide["pred_event_id_2"] = -1

wide = wide[
    ["asset_id", "physical_order_index", "pred_event_id_1", "pred_event_id_2"]
]

wide["pred_event_id_1"] = wide["pred_event_id_1"].fillna(-1).astype(np.int16)
wide["pred_event_id_2"] = wide["pred_event_id_2"].fillna(-1).astype(np.int16)

df_model = df_model.merge(
    wide,
    on=["asset_id", "physical_order_index"],
    how="left"
)

df_model["pred_event_id_1"] = df_model["pred_event_id_1"].fillna(-1).astype(np.int16)
df_model["pred_event_id_2"] = df_model["pred_event_id_2"].fillna(-1).astype(np.int16)

print("Rows with prediction event:")
print(
    (
        df_model["pred_event_id_1"].ne(-1)
        | df_model["pred_event_id_2"].ne(-1)
    ).sum()
)

print("\nPrediction event IDs:")
print(
    sorted(
        set(df_model.loc[df_model["pred_event_id_1"].ne(-1), "pred_event_id_1"].astype(int))
        | set(df_model.loc[df_model["pred_event_id_2"].ne(-1), "pred_event_id_2"].astype(int))
    )
)

Rows with prediction event:
69103

Prediction event IDs:
[2, 7, 19, 21, 23, 27, 34, 52, 53, 74, 77, 82, 83, 86, 87]


In [ ]:
# =========================================================
# 14. Build windows from df_model
# =========================================================

df_model = df_model.sort_values(["asset_id", "physical_order_index"]).reset_index(drop=True)

WINDOW = 24
STRIDE = 12
EXPECTED_STEP = np.timedelta64(10, "m")

X_all = df_model[MODEL_FEATURE_COLS].to_numpy(dtype=np.float32)
time_all = pd.to_datetime(df_model["reconstructed_time"]).to_numpy()
asset_all = df_model["asset_id"].to_numpy()
split_all = df_model["split_code"].to_numpy()
pred1_all = df_model["pred_event_id_1"].to_numpy()
pred2_all = df_model["pred_event_id_2"].to_numpy()


def build_windows_from_mask(mask):
    X_list = []
    meta = []

    for aid in np.unique(asset_all[mask]):
        pos = np.flatnonzero(mask & (asset_all == aid))

        if len(pos) < WINDOW:
            continue

        for i in range(0, len(pos) - WINDOW + 1, STRIDE):
            wpos = pos[i:i + WINDOW]

            if not np.all(np.diff(time_all[wpos]) == EXPECTED_STEP):
                continue

            X_list.append(X_all[wpos])
            meta.append({
                "start_row": int(wpos[0]),
                "end_row": int(wpos[-1]),
            })

    if len(X_list) == 0:
        return np.empty((0, WINDOW, len(MODEL_FEATURE_COLS)), dtype=np.float32), pd.DataFrame(meta)

    return np.stack(X_list).astype(np.float32), pd.DataFrame(meta)


X_train, train_window_meta = build_windows_from_mask(split_all == 0)
X_val, val_window_meta = build_windows_from_mask(split_all == 1)

event_ids = sorted(
    set(pred1_all[pred1_all != -1].astype(int))
    | set(pred2_all[pred2_all != -1].astype(int))
)

X_pred_by_event = {}
pred_meta_by_event = {}

for eid in event_ids:
    event_mask = (pred1_all == eid) | (pred2_all == eid)
    X_eid, meta_eid = build_windows_from_mask(event_mask)

    X_pred_by_event[eid] = X_eid
    pred_meta_by_event[eid] = meta_eid

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("\nPrediction windows:")
for eid in event_ids:
    print(eid, X_pred_by_event[eid].shape)

X_train: (30591, 24, 49)
X_val: (3378, 24, 49)

Prediction windows:
2 (183, 24, 49)
7 (443, 24, 49)
19 (311, 24, 49)
21 (107, 24, 49)
23 (164, 24, 49)
27 (827, 24, 49)
34 (335, 24, 49)
52 (227, 24, 49)
53 (503, 24, 49)
74 (255, 24, 49)
77 (767, 24, 49)
82 (203, 24, 49)
83 (1175, 24, 49)
86 (242, 24, 49)
87 (251, 24, 49)


In [ ]:
# =========================================================
# 15. Save model-ready outputs
# =========================================================

import joblib

MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)

df_model_path = MODEL_READY_DIR / "df_model.parquet"
feature_list_path = MODEL_READY_DIR / "model_ready_feature_list.json"
window_bundle_path = MODEL_READY_DIR / "windows_4h_stride2h_bundle.pkl"
pca_explained_path = MODEL_READY_DIR / "pca_explained_variance.csv"
pca_loadings_path = MODEL_READY_DIR / "pca_loadings.csv"
sample_path = MODEL_READY_DIR / "model_ready_sample.csv"

# Save df_model
df_model.to_parquet(df_model_path, index=False)

# Save feature list
with open(feature_list_path, "w") as f:
    json.dump(MODEL_FEATURE_COLS, f, indent=2)

# Save PCA summary for EDA
pca_explained = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(N_PC)],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
})

pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=MODEL_FEATURE_COLS,
    columns=[f"PC{i+1}" for i in range(N_PC)]
)

pca_explained.to_csv(pca_explained_path, index=False)
pca_loadings.to_csv(pca_loadings_path)

# Save small display sample
df_model.head(500).to_csv(sample_path, index=False)

# Save window bundle
bundle = {
    "df_model": df_model,
    "KMEANS_COLS": MODEL_FEATURE_COLS,
    "MODEL_FEATURE_COLS": MODEL_FEATURE_COLS,

    "X_train": X_train,
    "X_val": X_val,

    "train_window_meta": train_window_meta,
    "val_window_meta": val_window_meta,

    "X_pred_by_event": X_pred_by_event,
    "pred_meta_by_event": pred_meta_by_event,

    "window": WINDOW,
    "stride": STRIDE,
}

joblib.dump(bundle, window_bundle_path, compress=3)

print("Saved:")
print(df_model_path)
print(feature_list_path)
print(window_bundle_path)
print(pca_explained_path)
print(pca_loadings_path)
print(sample_path)

Saved:
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/df_model.parquet
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/model_ready_feature_list.json
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/windows_4h_stride2h_bundle.pkl
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/pca_explained_variance.csv
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/pca_loadings.csv
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/model_ready_sample.csv


In [ ]:
# =========================================================
# 16. Final consistency summary
# =========================================================

summary = {
    "df_model_shape": list(df_model.shape),
    "n_model_features": len(MODEL_FEATURE_COLS),
    "n_pca_metadata": len(PCA_META_COLS),
    "n_kmeans_metadata": len(KMEANS_META_COLS),
    "X_train_shape": list(X_train.shape),
    "X_val_shape": list(X_val.shape),
    "prediction_event_ids": [int(eid) for eid in event_ids],
    "prediction_window_shapes": {
        str(eid): list(X_pred_by_event[eid].shape)
        for eid in event_ids
    },
}

summary_path = MODEL_READY_DIR / "preprocessing_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved summary:", summary_path)

{
  "df_model_shape": [
    560060,
    71
  ],
  "n_model_features": 49,
  "n_pca_metadata": 11,
  "n_kmeans_metadata": 2,
  "X_train_shape": [
    30591,
    24,
    49
  ],
  "X_val_shape": [
    3378,
    24,
    49
  ],
  "prediction_event_ids": [
    2,
    7,
    19,
    21,
    23,
    27,
    34,
    52,
    53,
    74,
    77,
    82,
    83,
    86,
    87
  ],
  "prediction_window_shapes": {
    "2": [
      183,
      24,
      49
    ],
    "7": [
      443,
      24,
      49
    ],
    "19": [
      311,
      24,
      49
    ],
    "21": [
      107,
      24,
      49
    ],
    "23": [
      164,
      24,
      49
    ],
    "27": [
      827,
      24,
      49
    ],
    "34": [
      335,
      24,
      49
    ],
    "52": [
      227,
      24,
      49
    ],
    "53": [
      503,
      24,
      49
    ],
    "74": [
      255,
      24,
      49
    ],
    "77": [
      767,
      24,
      49
    ],
    "82": [
      203,
      24,
      49
    ],
    "83

In [ ]:
# Save EDA base table for 03_eda_normal_operating_space.ipynb
# This keeps pre-pruned columns for density plots, IQR diagnostics, and correlation analysis.
eda_base_path = MODEL_READY_DIR / "eda_base.parquet"
d.to_parquet(eda_base_path, index=False)

print("Saved EDA base:")
print(eda_base_path)

Saved EDA base:
/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v3/model_ready_simple/eda_base.parquet
